In [ ]:
import numpy as np
import rasterio as rio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from pathlib import Path

In [ ]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
FEATURE_DIR = RESULTS / "features"

In [3]:
country_raster_1km_  = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')

In [7]:
D2water_ = rio.open(DATA / 'distance2water_30arcsec.tif')

In [8]:
D2water = D2water_.read(1)

In [9]:
D2water = np.where(D2water < 0, np.nan, D2water).astype(np.float32)

In [10]:
D2water_1km = np.empty(country_raster_1km_.shape, dtype=np.float32)
reproject(
    source=D2water,
    destination=D2water_1km,
    src_transform=D2water_.transform,
    src_crs=D2water_.crs,
    dst_transform=country_raster_1km_.transform,
    dst_crs=country_raster_1km_.crs,
    resampling=Resampling.nearest,
)

(array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]], shape=(16922, 33844), dtype=float32),
 Affine(1000.0, 0.0, -16922000.0,
        0.0, -1000.0, 8461000.0))

In [11]:
with rio.open(
    FEATURE_DIR / 'distance-to-water-1km.tif',
    "w",
    driver="GTiff",
    height=D2water_1km.shape[0],
    width=D2water_1km.shape[1],
    count=1,
    dtype=D2water_1km.dtype,
    crs=country_raster_1km_.crs,
    transform=country_raster_1km_.transform,
    compress='lzw'
) as dst:
    dst.write(D2water_1km, 1)